# Tiếp tục A3 trên Colab với checkpoint trên Drive
A3 đã hoàn tất 3/9 epoch trên TRAIN90. Gói phục hồi giữ đúng source, cấu hình FP32/batch128/seed42, confidence TRAIN80 và checkpoint đã kiểm trên D.
Kết nối Drive và ghép/kiểm gói trên CPU trước. Sau khi chọn T4, chạy lại các cell chuẩn bị trong runtime mới. Mỗi lần khởi chạy chỉ thực hiện một lượt có hạn giờ, sao lưu theo SHA vào Drive sau khi worker kết thúc. Kiểm controller-result và BACKUP_COMPLETE trước lần tiếp tục.
Gói hiện hành là COLAB_RESUME.zip trong thư mục KLTN_AKT_RECOVERY của tài khoản Drive đang mount; SHA được ghim trong cell bootstrap.


In [ ]:
from pathlib import Path
from google.colab import drive
import json,sys,shutil
print(json.dumps(dict(status="DRIVE_MOUNT_REQUEST",python=sys.version,free_disk_bytes=shutil.disk_usage("/content").free)),flush=True)
drive.mount("/content/drive")
mydrive=Path("/content/drive/MyDrive")
assert mydrive.is_dir(), "Drive chua duoc mount"
folder=mydrive/"KLTN_AKT_RECOVERY"
assert not folder.is_symlink()
folder.mkdir(exist_ok=True)
print(json.dumps(dict(status="DRIVE_MOUNTED",folder=str(folder),zip_files=[p.name for p in folder.glob("*.zip")],training_started=False)))


In [ ]:
"""Kiểm và giải gói A3 trên Colab, giữ nguyên snapshot số học đã đóng."""
import hashlib
import json
from pathlib import Path
import shutil
import zipfile


def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def checked_destination(root, name):
    destination = (root / name).resolve()
    if not destination.is_relative_to(root.resolve()):
        raise ValueError('Member vượt thư mục đích')
    return destination


def copy_member(saved, member, destination, pin=None):
    if destination.exists():
        if pin and destination.stat().st_size == pin['bytes'] and digest(destination) == pin['sha256']:
            return
        raise FileExistsError(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with saved.open(member) as source, destination.open('xb') as target:
        shutil.copyfileobj(source, target)
    if pin and (destination.stat().st_size != pin['bytes'] or digest(destination) != pin['sha256']):
        raise ValueError('Member không khớp inventory ' + member)


def bootstrap(index, archive_path=None):
    transfer = Path('/content/kltn-a3-transfer')
    archive = Path(archive_path) if archive_path is not None else transfer / index['archive']
    assert archive.stat().st_size == index['archive_bytes']
    assert digest(archive) == index['archive_sha256']
    root = Path(index['project_root'])
    source_zip = transfer / 'SOURCE.zip'
    with zipfile.ZipFile(archive) as saved:
        expected = {'SOURCE.zip', *index['entries']}
        assert set(saved.namelist()) == expected
        assert len(saved.namelist()) == len(expected)
        copy_member(saved, 'SOURCE.zip', source_zip,
                    dict(bytes=saved.getinfo('SOURCE.zip').file_size, sha256=index['source_zip_sha256']))
        with zipfile.ZipFile(source_zip) as source:
            inventory = json.loads(source.read('INPUTS.json'))
            assert set(source.namelist()) == {'INPUTS.json', 'CPU_CHECK.json',
                                             *('project/' + name for name in inventory['files'])}
            assert json.loads(source.read('CPU_CHECK.json'))['status'] == 'PASS_LOCAL_STAGE_INPUTS'
            for name, pin in inventory['files'].items():
                copy_member(source, 'project/' + name, checked_destination(root, name), pin)
        for name, pin in index['entries'].items():
            assert name.startswith('recovery/')
            copy_member(saved, name, checked_destination(root, name.removeprefix('recovery/')), pin)
    assert digest(root / index['resume_checkpoint']) == index['resume_sha256']
    (transfer / 'resume-index.json').write_text(json.dumps(index,indent=2))
    proof = dict(status='PASS_COLAB_A3_INPUT_SHA256', project_root=str(root),
                 source_files=len(inventory['files']), recovery_files=len(index['entries']),
                 checkpoint_sha256=index['resume_sha256'], complete_epochs=index['complete_epochs'],
                 partial_epoch=index['partial_epoch'], partial_optimizer_step=index['partial_optimizer_step'],
                 training_started=False)
    (transfer / 'bootstrap-verified.json').write_text(json.dumps(proof, indent=2))
    print(json.dumps(proof))

INDEX = {'schema': 'a3-portable-refit-recovery-v1', 'status': 'RESUMABLE_PARTIAL_REFIT', 'archive': 'COLAB_RESUME.zip', 'archive_bytes': 683321739, 'archive_sha256': 'af20ed588be0cfe5c357602aac344c446cab7b38570f5fa7fbea1e45ce31bee9', 'source_zip_sha256': 'c3a109b4c07bcd7d802ccc35938a9d94be8d36377090fe08d6f1426ea523938a', 'project_root': '/opt/kltn', 'complete_epochs': 3, 'partial_epoch': None, 'partial_optimizer_step': None, 'frozen_epochs': 9, 'resume_checkpoint': 'experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-003/state.pt', 'resume_sha256': 'c9936d5e42dbbdbeb7a0c2fb7c80b8616e363cbfd091169e9d9ca448b8e2baed', 'entries': {'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-001/state.pt': {'bytes': 28913759, 'sha256': 'd0f60d41638bd9263275327f2740fc72b037a9938e9609755dc843cece9a9c34'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-001/metadata.json': {'bytes': 2286, 'sha256': '7d68037dd267a49d35efb90ae218b45b9a0c168555fa8280e8bb017352e6d8f3'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-001/COMPLETE.json': {'bytes': 183, 'sha256': 'c6797784fc601ab90b7cb91d2b7e3d36d86776525be52ea840ca073b43e4054a'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-002/state.pt': {'bytes': 28921503, 'sha256': 'ad432d11a230c38aac32d56ace419e5c9cf212e035a1dbbd7c9851ee387041d3'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-002/metadata.json': {'bytes': 2286, 'sha256': 'f02d067754186ebab919d278bc5e43f2c0d824e99952c664d36707cef3a45f3b'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-002/COMPLETE.json': {'bytes': 183, 'sha256': 'ab14943816fc00286d5e3a002b9532f0dcc08e20f8a01166eab5ebcf7df5cb6b'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-003/state.pt': {'bytes': 28929119, 'sha256': 'c9936d5e42dbbdbeb7a0c2fb7c80b8616e363cbfd091169e9d9ca448b8e2baed'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-003/metadata.json': {'bytes': 2286, 'sha256': '9f6e99c851c226818c9f47c3657c1e2a671f5c75eb663f783d27ff0aef87ad3b'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-003/COMPLETE.json': {'bytes': 183, 'sha256': '10498f92fa1fc6a407228b31b7b2878bb1ccf64cb16a36bd2692254fef4d46f0'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/run-manifest.json': {'bytes': 2055, 'sha256': '7342265521164c5415858558de414cc683e5f427fc697f6701324f6aa074de44'}}, 'source_receipt': 'experiments/akt/results/modal-junyi-a3-refit-l4-20261115-05/recovery/remote-inventory.json', 'next_device_verified': False, 'manifest_source': 'experiments/akt/results/modal-junyi-a3-refit-l4-20261115-03/files/outputs/junyi-a3-refit-20261114-01/run-manifest.json', 'limits_vi': 'Giữ model/Adam/RNG/sampler và history. Còn học tiếp đến đủ9epoch. Phải kiểm source, runtime và thiết bị đích trước resume. Chưa upload hoặc chạy trên Colab.'}
archive=Path('/content/drive/MyDrive/KLTN_AKT_RECOVERY/COLAB_RESUME.zip')
assert archive.is_file(), 'Can upload ZIP vao Drive tai khoan dang mount'
bootstrap(INDEX,str(archive))
print('PASS_PERSONAL_DRIVE_BOOTSTRAP')


In [ ]:
import json, subprocess, sys, time
from pathlib import Path
root=Path("/opt/kltn")
assert Path("/content/kltn-a3-transfer/bootstrap-verified.json").is_file()
venv=root/".venv"
assert not venv.exists(), "Moi truong da ton tai, can kiem truoc khi dung lai"
started=time.monotonic()
subprocess.run([sys.executable,"-m","venv","--without-pip",str(venv)],check=True,timeout=60)
log=Path("/content/kltn-a3-transfer/runtime-install.log")
with log.open("w") as output:
    subprocess.run([sys.executable,"-m","pip","--python",str(venv/"bin/python"),"install","--require-hashes","--no-deps","--only-binary=:all:","--no-cache-dir","--retries","1","--timeout","20","-r",str(root/"artifacts/akt/runtime/modal-linux-cuda.lock.txt")],stdout=output,stderr=subprocess.STDOUT,check=True,timeout=540)
probe=subprocess.run([str(venv/"bin/python"),"-B","-c","import json,torch; assert torch.cuda.is_available(); print(json.dumps(dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory)))"],capture_output=True,text=True,check=True,timeout=60)
report=dict(status="PASS_ISOLATED_CUDA_RUNTIME",seconds=time.monotonic()-started,device=json.loads(probe.stdout),global_install=False)
Path("/content/kltn-a3-transfer/runtime-verified.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report))


In [ ]:
BACKUP_SOURCE = '"""Sao lưu checkpoint Colab theo SHA, xuất manifest trước khi cập nhật LATEST."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport uuid\n\n\ndef sha(path):\n    with path.open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef position(folder):\n    full = re.fullmatch(r\'epoch-(\\d{3})\', folder.name)\n    if full:\n        return int(full[1]), 2040, 1\n    partial = re.fullmatch(r\'refit-batch-epoch-(\\d+)-step-(\\d+)-[a-f0-9]+\', folder.name)\n    if partial:\n        return int(partial[1]), int(partial[2]), 0\n    raise ValueError(\'Tên checkpoint không hợp lệ\')\n\n\ndef restore(root, backup_root):\n    root, backup_root = Path(root), Path(backup_root)\n    pointer = json.loads((backup_root/\'LATEST.json\').read_text())\n    manifest = (backup_root/pointer[\'manifest\']).resolve()\n    assert manifest.is_relative_to(backup_root.resolve())\n    assert sha(manifest) == pointer[\'sha256\']\n    receipt = json.loads(manifest.read_text())\n    assert receipt[\'status\'] == \'PASS_DRIVE_CHECKPOINT_BACKUP\'\n    root.mkdir(parents=True,exist_ok=True)\n    for relative,pin in receipt[\'files\'].items():\n        source = (backup_root/pin[\'drive_relative\']).resolve()\n        destination = (root/relative).resolve()\n        assert source.is_relative_to(backup_root.resolve()) and destination.is_relative_to(root.resolve())\n        assert source.stat().st_size == pin[\'bytes\'] and sha(source) == pin[\'sha256\']\n        destination.parent.mkdir(parents=True,exist_ok=True)\n        if destination.exists():\n            assert destination.stat().st_size == pin[\'bytes\'] and sha(destination) == pin[\'sha256\']\n            continue\n        temporary = root/(\'.restoring-\'+uuid.uuid4().hex)\n        with source.open(\'rb\') as origin, temporary.open(\'xb\') as target:\n            shutil.copyfileobj(origin,target)\n        assert temporary.stat().st_size == pin[\'bytes\'] and sha(temporary) == pin[\'sha256\']\n        temporary.replace(destination)\n    assert sha(root/receipt[\'checkpoint\']) == receipt[\'checkpoint_sha256\']\n    return receipt\n\n\ndef backup(root, operation, backup_root):\n    root, operation, backup_root = map(Path, (root, operation, backup_root))\n    execution = json.loads((operation/\'execution.json\').read_text())\n    worker = json.loads((root/\'gpu-stage-request.json\').read_text())\n    logical = root/\'experiments/akt/results\'/worker[\'logical_run_id\']\n    folders = [marker.parent for marker in (logical/\'checkpoints\').glob(\'*/COMPLETE.json\')]\n    latest = max(folders, key=position)\n    selected = [latest, *(folder for folder in folders if folder.name.startswith(\'epoch-\') and folder != latest)]\n    files = []\n    for folder in selected:\n        marker = json.loads((folder/\'COMPLETE.json\').read_text())\n        metadata = json.loads((folder/\'metadata.json\').read_text())\n        assert sha(folder/\'metadata.json\') == marker[\'metadata_sha256\']\n        assert metadata[\'checkpoint_sha256\'] == marker[\'checkpoint_sha256\']\n        assert (folder/\'state.pt\').stat().st_size == metadata[\'checkpoint_bytes\']\n        assert sha(folder/\'state.pt\') == marker[\'checkpoint_sha256\']\n        files.extend(folder/name for name in (\'state.pt\',\'metadata.json\',\'COMPLETE.json\'))\n    files.extend(path for path in logical.iterdir() if path.is_file() and path.suffix in (\'.json\',\'.jsonl\'))\n    files.extend(path for path in operation.iterdir() if path.is_file()\n                 and path.suffix in (\'.json\',\'.log\') and path.name != \'controller-result.json\')\n    entries = {}\n    for source in files:\n        relative = source.relative_to(root).as_posix()\n        if \'/checkpoints/epoch-\' in relative:\n            drive_relative = \'completed/\'+relative\n        else:\n            drive_relative = \'tranches/\'+execution[\'execution_id\']+\'/files/\'+relative\n        destination = backup_root/drive_relative\n        pin = dict(bytes=source.stat().st_size, sha256=sha(source), drive_relative=drive_relative)\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        if not destination.exists():\n            temporary = backup_root/(\'.uploading-\'+uuid.uuid4().hex)\n            with source.open(\'rb\') as origin, temporary.open(\'xb\') as target:\n                shutil.copyfileobj(origin,target)\n            assert temporary.stat().st_size == pin[\'bytes\'] and sha(temporary) == pin[\'sha256\']\n            temporary.replace(destination)\n        assert destination.stat().st_size == pin[\'bytes\'] and sha(destination) == pin[\'sha256\']\n        entries[relative] = pin\n    epoch, step, full = position(latest)\n    state = latest/\'state.pt\'\n    assert sha(state) == json.loads((latest/\'COMPLETE.json\').read_text())[\'checkpoint_sha256\']\n    receipt = dict(status=\'PASS_DRIVE_CHECKPOINT_BACKUP\', execution_id=execution[\'execution_id\'],\n        checkpoint=state.relative_to(root).as_posix(), checkpoint_sha256=sha(state),\n        complete_epochs=epoch if full else epoch-1, partial_epoch=None if full else epoch,\n        partial_optimizer_step=None if full else step, frozen_epochs=worker[\'frozen_epochs\'],\n        refit_complete=(logical/\'COMPLETE.json\').exists(), files=entries)\n    manifest = backup_root/\'tranches\'/execution[\'execution_id\']/\'BACKUP_COMPLETE.json\'\n    manifest.parent.mkdir(parents=True, exist_ok=True)\n    if manifest.exists():\n        assert json.loads(manifest.read_text()) == receipt\n    else:\n        with manifest.open(\'x\') as stream:\n            json.dump(receipt,stream,indent=2)\n    assert json.loads(manifest.read_text()) == receipt\n    pointer = backup_root/\'LATEST.json\'\n    if pointer.exists():\n        previous = json.loads(pointer.read_text())\n        previous_manifest = (backup_root/previous[\'manifest\']).resolve()\n        assert previous_manifest.is_relative_to(backup_root.resolve())\n        assert sha(previous_manifest) == previous[\'sha256\']\n        previous_receipt = json.loads(previous_manifest.read_text())\n        previous_folder = Path(previous_receipt[\'checkpoint\']).parent\n        if position(previous_folder) > position(latest):\n            return manifest\n    temporary = backup_root/(\'LATEST.uploading-\'+uuid.uuid4().hex+\'.json\')\n    temporary.write_text(json.dumps(dict(manifest=manifest.relative_to(backup_root).as_posix(),\n                                         sha256=sha(manifest)),indent=2))\n    temporary.replace(pointer)\n    return manifest\n'
CONTROLLER_SOURCE = '"""Một lượt tiếp tục A3 có hạn giờ trên T4 miễn phí, dùng worker đã đóng."""\nimport argparse\nfrom contextlib import contextmanager\nimport csv\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport threading\nimport time\n\nROOT = Path(\'/opt/kltn\')\nTRANSFER = Path(\'/content/kltn-a3-transfer\')\n\n\ndef sha(path):\n    with path.open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\n@contextmanager\ndef measure_gpu(operation):\n    """NVML qua nvidia-smi, không thay Torch/RNG hoặc trọng số."""\n    stop = threading.Event()\n    started = time.monotonic()\n    def sample():\n        with (operation/\'gpu-telemetry.jsonl\').open(\'x\') as log:\n            while not stop.is_set():\n                try:\n                    query = subprocess.run([\'nvidia-smi\',\n                        \'--query-gpu=utilization.gpu,memory.used,memory.total,power.draw\',\n                        \'--format=csv,noheader,nounits\'],capture_output=True,text=True,check=True,timeout=5)\n                    values = next(csv.reader(query.stdout.splitlines()))\n                    event = dict(seconds=time.monotonic()-started,gpu_utilization=float(values[0]),\n                        vram_used_mib=float(values[1]),vram_total_mib=float(values[2]),power_w=float(values[3]))\n                except Exception as error:\n                    event = dict(seconds=time.monotonic()-started,error_type=type(error).__name__,error=str(error))\n                log.write(json.dumps(event)+\'\\n\')\n                log.flush()\n                if stop.wait(5):\n                    break\n    thread = threading.Thread(target=sample,daemon=True)\n    thread.start()\n    try:\n        yield\n    finally:\n        stop.set()\n        thread.join(timeout=10)\n        assert not thread.is_alive(), \'GPU sampler chưa kết thúc\'\n\n\ndef run(attempt, checkpoint, checkpoint_sha, backup_root=None):\n    assert json.loads((TRANSFER / \'runtime-verified.json\').read_text())[\'status\'] == \'PASS_ISOLATED_CUDA_RUNTIME\'\n    if backup_root is not None:\n        backup_root = Path(backup_root)\n        assert backup_root.resolve().is_relative_to(Path(\'/content/drive/MyDrive\').resolve())\n        assert backup_root.is_dir(), \'Drive chưa sẵn sàng để lưu checkpoint\'\n    worker = ROOT / \'gpu-stage-request.json\'\n    frozen = json.loads(worker.read_text())\n    assert frozen[\'variant\'] == \'A3\' and frozen[\'frozen_epochs\'] == 9\n    logical = ROOT / \'experiments/akt/results\' / frozen[\'logical_run_id\']\n    resume = (ROOT / checkpoint).resolve()\n    assert resume.is_relative_to((logical / \'checkpoints\').resolve())\n    assert sha(resume) == checkpoint_sha\n    assert not (logical / \'COMPLETE.json\').exists(), \'Refit đã hoàn tất, không chạy thêm\'\n    execution_id = f\'colab-junyi-a3-refit-20261115-{attempt:03}\'\n    operation = ROOT / \'experiments/akt/results\' / (execution_id + \'-operation\')\n    operation.mkdir(exist_ok=False)\n    request = dict(execution_id=execution_id, gpu=\'T4\', worker_request_sha256=sha(worker),\n                   approval_reference=\'Người dùng15/11 duyệt tiếp tục A3 bằng T4 Colab miễn phí và chuyển gói683MB.\',\n                   resume_checkpoint=checkpoint, resume_sha256=checkpoint_sha)\n    execution = operation / \'execution.json\'\n    execution.write_text(json.dumps(request, ensure_ascii=False, indent=2))\n    lock = TRANSFER / \'training.lock\'\n    lock.parent.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    with lock.open(\'x\') as stream:\n        json.dump(dict(pid=os.getpid(), execution_id=execution_id), stream)\n    result = dict(status=\'FAILED\', execution_id=execution_id, checkpoint_input_sha256=checkpoint_sha)\n    process = None\n    try:\n        with measure_gpu(operation), (operation / \'console.log\').open(\'x\') as output:\n            process = subprocess.Popen([str(ROOT / \'.venv/bin/python\'), \'-u\', \'-B\',\n                str(ROOT / \'scripts/akt/run_junyi_refit_gpu_stage.py\'), \'--request\', str(worker),\n                \'--execution\', str(execution)], cwd=ROOT, stdout=output, stderr=subprocess.STDOUT,\n                start_new_session=True, env={**os.environ, \'PYTHONDONTWRITEBYTECODE\': \'1\'})\n            (operation / \'child.json\').write_text(json.dumps(dict(pid=process.pid)))\n            try:\n                code = process.wait(timeout=600)\n            except subprocess.TimeoutExpired:\n                os.killpg(process.pid, signal.SIGTERM)\n                try:\n                    process.wait(timeout=8)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGKILL)\n                    process.wait(timeout=10)\n                raise TimeoutError(\'Dừng lượt600giây; giữ checkpoint gần nhất, cần kiểm trước tiếp tục\')\n        result.update(exit_code=code)\n        assert code == 0, \'Worker lỗi, đọc console/failure trước khi tiếp tục\'\n        native = json.loads((operation / \'result.json\').read_text())\n        assert native[\'status\'] in {\'PARTIAL_JUNYI_REFIT90\', \'JUNYI_REFIT90_COMPLETE\'}, native[\'status\']\n        result.update(status=\'PASS_BOUNDED_NATIVE_WORKER\', native_status=native[\'status\'])\n        if backup_root is not None:\n            from colab_a3_backup import backup\n            result[\'drive_backup\'] = str(backup(ROOT,operation,backup_root))\n    except BaseException:\n        if process is not None and process.poll() is None:\n            os.killpg(process.pid, signal.SIGTERM)\n            try:\n                process.wait(timeout=8)\n            except subprocess.TimeoutExpired:\n                os.killpg(process.pid, signal.SIGKILL)\n                process.wait(timeout=10)\n        raise\n    finally:\n        result[\'wall_seconds\'] = time.monotonic() - started\n        result[\'child_running\'] = process is not None and process.poll() is None\n        (operation / \'controller-result.json\').write_text(json.dumps(result, ensure_ascii=False, indent=2))\n        if not result[\'child_running\']:\n            lock.unlink()\n        print(json.dumps(result, ensure_ascii=False), flush=True)\n    return result\n\n\ndef run_sequence(attempt, checkpoint, checkpoint_sha, backup_root, max_tranches, sequence_seconds):\n    """Tiếp tục một protocol đã đóng, dừng ở9epoch/lỗi/giới hạn và kiểm backup mỗi lượt."""\n    from colab_a3_backup import position\n    assert backup_root is not None and 1 <= max_tranches <= 40\n    assert 650 <= sequence_seconds <= 10800\n    started = time.monotonic()\n    backup_root = Path(backup_root)\n    previous = position(Path(checkpoint).parent)\n    state = dict(status=\'RUNNING\', complete_epochs=previous[0] if previous[2] else previous[0]-1)\n    try:\n        for offset in range(max_tranches):\n            if time.monotonic()-started+620 > sequence_seconds:\n                state[\'status\'] = \'TIME_LIMIT_CHECKPOINT_SAVED\'\n                break\n            result = run(attempt+offset,checkpoint,checkpoint_sha,backup_root)\n            assert result[\'status\'] == \'PASS_BOUNDED_NATIVE_WORKER\' and not result[\'child_running\']\n            manifest = Path(result[\'drive_backup\'])\n            receipt = json.loads(manifest.read_text())\n            assert receipt[\'status\'] == \'PASS_DRIVE_CHECKPOINT_BACKUP\' and receipt[\'frozen_epochs\'] == 9\n            pointer = json.loads((backup_root/\'LATEST.json\').read_text())\n            assert (backup_root/pointer[\'manifest\']).resolve() == manifest.resolve()\n            assert sha(manifest) == pointer[\'sha256\']\n            current = position(Path(receipt[\'checkpoint\']).parent)\n            assert current > previous, \'Không có checkpoint tiến triển, dừng trước lượt tiếp theo\'\n            checkpoint, checkpoint_sha = receipt[\'checkpoint\'], receipt[\'checkpoint_sha256\']\n            previous = current\n            state.update(complete_epochs=receipt[\'complete_epochs\'],partial_epoch=receipt[\'partial_epoch\'],\n                partial_optimizer_step=receipt[\'partial_optimizer_step\'],checkpoint=checkpoint,\n                checkpoint_sha256=checkpoint_sha,last_attempt=attempt+offset,\n                status=\'COMPLETE_9_EPOCHS_DRIVE_BACKED_UP\' if receipt[\'refit_complete\'] else \'RUNNING\')\n            if receipt[\'refit_complete\']:\n                assert receipt[\'complete_epochs\'] == 9\n                break\n        else:\n            state[\'status\'] = \'TRANCHE_LIMIT_CHECKPOINT_SAVED\'\n    except BaseException as error:\n        state.update(status=\'STOPPED_REQUIRES_REVIEW\',error_type=type(error).__name__,error=str(error))\n        raise\n    finally:\n        state[\'elapsed_seconds\'] = time.monotonic()-started\n        (TRANSFER/\'sequence-status.json\').write_text(json.dumps(state,ensure_ascii=False,indent=2))\n        temporary = backup_root/\'SEQUENCE_STATUS.writing.json\'\n        temporary.write_text(json.dumps(state,ensure_ascii=False,indent=2))\n        temporary.replace(backup_root/\'SEQUENCE_STATUS.json\')\n        print(json.dumps(state,ensure_ascii=False),flush=True)\n    return state\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--attempt\', type=int, required=True)\n    parser.add_argument(\'--checkpoint\', required=True)\n    parser.add_argument(\'--sha256\', required=True)\n    parser.add_argument(\'--backup-root\', type=Path)\n    parser.add_argument(\'--max-tranches\', type=int, default=1)\n    parser.add_argument(\'--sequence-seconds\', type=int, default=10800)\n    args = parser.parse_args()\n    assert args.attempt > 0\n    if args.max_tranches == 1:\n        run(args.attempt, args.checkpoint, args.sha256, args.backup_root)\n    else:\n        run_sequence(args.attempt,args.checkpoint,args.sha256,args.backup_root,\n                     args.max_tranches,args.sequence_seconds)\n'
from pathlib import Path
import importlib.util,json,subprocess
transfer=Path('/content/kltn-a3-transfer')
backup_root=Path('/content/drive/MyDrive/KLTN_AKT_RECOVERY')
assert backup_root.is_dir()
(transfer/'colab_a3_backup.py').write_text(BACKUP_SOURCE)
controller=transfer/'colab_a3_tranche.py'
controller.write_text(CONTROLLER_SOURCE)
index=json.loads((transfer/'resume-index.json').read_text())
if (backup_root/'LATEST.json').exists():
    spec=importlib.util.spec_from_file_location('a3_backup',transfer/'colab_a3_backup.py')
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    receipt=module.restore('/opt/kltn',backup_root)
    index.update(resume_checkpoint=receipt['checkpoint'],resume_sha256=receipt['checkpoint_sha256'])
assert not (transfer/'training.lock').exists(), 'Co luot dang chay, kiem truoc khi launch'
results=Path('/opt/kltn/experiments/akt/results')
existing=list(results.glob('colab-junyi-a3-refit-20261115-*-operation'))
attempt=max([int(p.name.split('-')[-2]) for p in existing],default=0)+1
log=transfer/f'launch-{attempt:03}.log'
with log.open('x') as output:
    process=subprocess.Popen(['/opt/kltn/.venv/bin/python','-u','-B',str(controller),'--attempt',str(attempt),'--checkpoint',index['resume_checkpoint'],'--sha256',index['resume_sha256'],'--backup-root',str(backup_root)],stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
print(json.dumps(dict(status='BOUNDED_T4_TRANCHE_LAUNCHED',attempt=attempt,pid=process.pid,log=str(log),maximum_worker_seconds=600,drive_backup_after_tranche=True)))
